# wp1pt4pt1f — per-site case studies & MSA analysis files

For each WP1 site this notebook:

1. designs a **3-storey MDOF CBF** for the site (Soil Class A, local `S_alpha,475`),
   in parallel, using the existing `standes` design algorithm;
2. computes the **design base-shear coefficient** `Vb_coeff = Vb / Wt` from each design;
3. builds the **MDOF analysis folder** (structural model + modal + MSA files) and runs a
   **modal analysis** to get the first-mode participation factor `Gamma` and equivalent
   SDOF mass `m_star`;
4. builds an **equivalent SDOF** per site from `Vb_coeff`, `m_star`, `Gamma`;
5. builds the **SDOF analysis folder** (structural model + MSA files) and the per-stripe
   ground-motion pickles for both GM sets;
6. writes a **batch launcher for the SDOFs only**.

Designs are saved **in-repo** under `casestudy_designs_site_specific`. Analysis folders are
written under `DEST_ROOT` (set below), organised `DEST_ROOT/site_{ii}/{sdof,mdof}/`.

In [1]:
%load_ext autoreload
%autoreload 2

## 0. Setup & parameters

In [2]:
import os
import re
import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from phd_project.config import config
from standes.utils import generate_type_1_tag

from phd_project.scripts.case_study_design_scripts.design_site_mdof import (
    design_sites_parallel,
    default_n_workers,
)
from phd_project.scripts.case_study_design_scripts.run_site_modal import (
    run_modal_analyses_parallel,
)
from phd_project.scripts.equivalent_sdof import convert_poc_to_eq_sdof_poc
from phd_project.scripts.loading_protocols import FEMA_461_loading_protocol
from phd_project.scripts.sdof_parameterisation import assemble_sdof_material_params
from phd_project.scripts.templates.copy_templates_to_folders import (
    copy_structural_model,
    copy_analysis_config,
    configure_batch_run_file,
    copy_file,
)

cfg = config.load_config()

In [3]:
# ----------------------------------------------------------------------------
# PARAMETERS
# ----------------------------------------------------------------------------
# Destination root for the (large) analysis folders -- set this to your external
# drive. Folders are written as DEST_ROOT/site_{ii}/sdof/ and .../mdof/.
DEST_ROOT = Path("C:/Users/clemettn/Desktop/test_folder")

# --- structure / design (fixed; matches 01_design_case_study_structures.py) ---
SITE_CATEGORY = "A"        # soil class for BOTH the MDOF design and SDOF spectrum
N_STOREYS = 3
DUCTILITY_CLASS = 2
STOREY_HEIGHT = 3500       # mm
BAY_WIDTH = 7000           # mm
ASPECT = STOREY_HEIGHT / BAY_WIDTH   # braced-bay aspect ratio (height / width)
MAX_DESIGN_ITERS = 15

# --- analysis ---
GM_SETS = ["AvgSA_03", "AvgSA_06"]
DAMPING_RATIO = 0.05
DISP_LIMIT = 700           # mm; SDOF collapse displacement limit
MDOF_DRIFT_LIMIT = 0.2     # MDOF collapse drift limit
MAX_N_RECORDS = None       # cap records per stripe (None = all)
STRIPE_ORDER_ASCENDING = True

# --- parallelism ---
N_WORKERS = default_n_workers()    # max(cpu_count - 3, 1)

# Restrict to a subset of sites while testing (e.g. [0, 1, 2]); None = all sites.
LIMIT_SITES = None

S_ALPHA_COLUMN = "S_alpha,475"

# --- cyclic pushover (FEMA 461) ---
CPO_U_MAX = 250          # mm, peak cyclic amplitude (+/-)
CPO_N_STEPS = 12         # FEMA 461 amplitude steps (12 -> matches example sequence)
CPO_DU = 0.2             # mm, base displacement ramp step (dU_max)
CPO_DISPLACEMENTS = np.round(
    np.append(FEMA_461_loading_protocol(CPO_U_MAX, CPO_N_STEPS), 0), 3
).tolist()
print(f"N_WORKERS = {N_WORKERS}; DEST_ROOT = {DEST_ROOT}")

N_WORKERS = 9; DEST_ROOT = C:\Users\clemettn\Desktop\test_folder


### Design-file helpers (mirrors `wp1pt4pt1a`)

In [4]:
def get_n_primary_modes_from_design_file(design_file_path: Path) -> int:
    """Number of primary horizontal modes = number of storeys (levels - 1)"""
    with open(design_file_path, "r") as f:
        design_data = json.load(f)
    return (len(design_data["structure"]["level_coordinates"]) - 1)


def get_n_damping_modes_from_design_file(design_file_path: Path) -> int:
    """Damp all horizontal modes (4 primary grid nodes per level)."""
    with open(design_file_path, "r") as f:
        design_data = json.load(f)
    return get_n_primary_modes_from_design_file(design_file_path) * len(design_data["structure"]["grid_coordinates"])


def _to_plain(params: dict) -> dict:
    """Coerce numpy scalars/arrays to plain Python floats/lists so they repr()
    cleanly into the generated structural_model.py."""
    out = {}
    for k, v in params.items():
        if isinstance(v, (list, tuple, np.ndarray)):
            out[k] = [float(x) for x in np.asarray(v).ravel()]
        else:
            out[k] = float(v)
    return out


def get_control_node_from_design_file(design_file_path: Path) -> int:
    """Roof node = top-left corner node tag (generate_type_1_tag over n_levels)."""
    with open(design_file_path, "r") as f:
        design_data = json.load(f)
    n_levels = len(design_data["structure"]["level_coordinates"])
    return generate_type_1_tag(1, 1, 1, n_levels, 0, 0)


## 1. Discover sites

Sites are discovered from the record-selection pickle filenames
(`site_{ii}__stripe_{n}__gm_selection.pickle`). The positional site index `ii`
maps to row `ii` of `sites.csv` (`.iloc[ii]`).

In [5]:
STRIPE_RE = re.compile(r"site_(\d+)__stripe_(\d+)__gm_selection\.pickle$")


def discover_site_stripes(gm_set: str) -> dict[int, list[Path]]:
    """Return {site_idx: [stripe pickle paths]} for a GM set."""
    folder = Path(cfg["results"][f"{gm_set}_record_selection"])
    out: dict[int, list[Path]] = {}
    for f in sorted(folder.iterdir()):
        m = STRIPE_RE.match(f.name)
        if m:
            out.setdefault(int(m.group(1)), []).append(f)
    return out


site_stripes = {g: discover_site_stripes(g) for g in GM_SETS}
all_sites = sorted(set().union(*[set(d) for d in site_stripes.values()]))
if LIMIT_SITES is not None:
    all_sites = [s for s in all_sites if s in set(LIMIT_SITES)]

sites_df = pd.read_csv(cfg["results"]["selected_sites_csv"])
site_S_alpha = {ii: float(sites_df.iloc[ii][S_ALPHA_COLUMN]) for ii in all_sites}
site_tag = {ii: f"{N_STOREYS}s_cbf_dc{DUCTILITY_CLASS}_site{ii}" for ii in all_sites}

print(f"{len(all_sites)} sites; S_alpha range "
      f"{min(site_S_alpha.values()):.3f} -> {max(site_S_alpha.values()):.3f}")

60 sites; S_alpha range 0.134 -> 1.132


## 2. Design the MDOFs (parallel + tqdm)

Each MDOF is designed with the existing `standes` algorithm (unchanged) at Soil
Class A and the site's `S_alpha,475`. Designs are written **in-repo**.

In [6]:
design_root = Path(cfg["models"]["casestudy_designs_site_specific"])
design_root.mkdir(parents=True, exist_ok=True)
summary_path = design_root / "site_designs_summary.csv"

jobs = [(site_S_alpha[ii], site_tag[ii], design_root) for ii in all_sites]

# only (re)design structures whose tag is not already in the summary csv
existing_df = pd.read_csv(summary_path, index_col="tag") if summary_path.exists() else None
done_tags = set(existing_df.index) if existing_df is not None else set()
pending_jobs = [j for j in jobs if j[1] not in done_tags]

if pending_jobs:
    print(f"designing {len(pending_jobs)} of {len(jobs)} structures "
          f"({len(jobs) - len(pending_jobs)} already in {summary_path.name})")
    design_results = design_sites_parallel(
        pending_jobs,
        n_workers=N_WORKERS,
        site_category=SITE_CATEGORY,
        ductility_class=DUCTILITY_CLASS,
        storey_height=STOREY_HEIGHT,
        n_storeys=N_STOREYS,
        max_iters=MAX_DESIGN_ITERS,
    )
    new_df = pd.DataFrame(design_results).set_index("tag")
    if existing_df is not None:
        existing_df = existing_df.drop(index=new_df.index, errors="ignore")
        designs_df = pd.concat([existing_df, new_df])
    else:
        designs_df = new_df
    designs_df = designs_df.sort_index()
    designs_df.to_csv(summary_path)
else:
    print(f"all {len(jobs)} structures already in {summary_path.name}; skipping design")
    designs_df = existing_df.sort_index()

designs = designs_df.to_dict("index")

failed = designs_df[~designs_df["success"].fillna(False)]
if len(failed):
    print(f"WARNING: {len(failed)} designs did not succeed:\n{failed.index.tolist()}")

designs_df[["S_alpha_RP", "Vb_coeff", "Wt", "T", "q_design", "success"]]

all 60 structures already in site_designs_summary.csv; skipping design


,S_alpha_RP,Vb_coeff,Wt,T,q_design,success
tag,,,,,,
3s_cbf_dc2_site0,0.209594,0.028540,3061800.0,0.719978,2.040000,True
3s_cbf_dc2_site1,0.273973,0.039123,3061800.0,0.686561,2.040000,True
3s_cbf_dc2_site10,0.394579,0.084847,3061800.0,0.581311,2.040000,True
3s_cbf_dc2_site11,0.355457,0.076434,3061800.0,0.581311,2.040000,True
3s_cbf_dc2_site12,0.287359,0.041914,3061800.0,0.672157,2.040000,True
3s_cbf_dc2_site13,0.281817,0.041105,3061800.0,0.672157,2.040000,True
3s_cbf_dc2_site14,0.313111,0.046007,3061800.0,0.667227,2.040000,True
3s_cbf_dc2_site15,0.319417,0.039983,3061800.0,0.639112,2.500000,True
3s_cbf_dc2_site16,0.244726,0.034946,3061800.0,0.686561,2.040000,True


## 3. Build the MDOF analysis folders (structural model + modal + MSA)

One MDOF folder per site at `DEST_ROOT/site_{ii}/mdof/`, populated with the
shared model + a modal config (for the participation factor) + the MSA files for
both GM sets. No MDOF batch file is generated.

In [7]:
def _stripe_pickles_for(site_idx: int, gm_set: str) -> list[Path]:
    return site_stripes[gm_set].get(site_idx, [])


def add_msa_files(folder: Path, site_idx: int, recorder_template_key: str):
    """Add the MSA run/coordinator/injection/recorder files + one config per GM
    set (each with its own stripe-pickle subfolder)."""
    copy_file(cfg["templates"]["run_msa_site"], folder / "run_msa_site.py")
    copy_file(cfg["templates"]["run_batch_msa_per_stripe_record"],
              folder / "run_batch_msa_per_stripe_record.py")
    copy_file(cfg["templates"]["nltha_injection_update_damping"],
              folder / "injection_functions.py")
    copy_file(cfg["templates"][recorder_template_key],
              folder / "msa_process_recorders.py")

    record_src = Path(cfg["proc_data"]["gm_records"]).as_posix()
    for gm_set in GM_SETS:
        gm_dir = folder / gm_set
        gm_dir.mkdir(parents=True, exist_ok=True)
        for src in _stripe_pickles_for(site_idx, gm_set):
            copy_file(src, gm_dir / src.name)
        copy_analysis_config(
            cfg["templates"]["config_msa"],
            folder / f"config_msa_{gm_set}.py",
            results_folder_name=f"msa_{gm_set}",
            gm_selection_src_str=gm_dir.as_posix(),
            record_src_str=record_src,
            stripe_order_ascending=STRIPE_ORDER_ASCENDING,
            max_n_records=MAX_N_RECORDS,
        )


def add_cyclic_pushover_files(folder: Path, ctrl_node: int) -> Path:
    """Add a FEMA 461 cyclic pushover run script + config to an analysis folder.
    The config imports structural_model/model_init from the folder itself."""
    copy_file(cfg["templates"]["run_cyclic_pushover"], folder / "run_cyclic_pushover.py")
    cfg_dst = folder / "config_cyclic_pushover.py"
    copy_analysis_config(
        cfg["templates"]["config_cyclic_pushover"],
        cfg_dst,
        results_folder_name="cyclic_pushover",
        update_config={
            "displacement_type": "displacement",
            "dU": CPO_DU,
            "ctrl_node": ctrl_node,
            "displacements": CPO_DISPLACEMENTS,
        },
    )
    return cfg_dst


def build_mdof_folder(site_idx: int) -> Path:
    tag = site_tag[site_idx]
    folder = DEST_ROOT / f"site_{site_idx}" / "mdof"
    folder.mkdir(parents=True, exist_ok=True)

    # design file (structural_model reads it from its own folder)
    design_out = Path(designs[tag]["design_out_json"])
    design_dst = folder / f"{tag}_designfile.json"
    copy_file(design_out, design_dst)

    # structural model
    n_damping_modes = get_n_damping_modes_from_design_file(design_dst)
    copy_structural_model(
        cfg["templates"]["structural_model"],
        folder / "structural_model.py",
        design_json=design_dst.name,
        damping_updates={"n_modes": n_damping_modes, "damping_ratio": DAMPING_RATIO},
        recorder_updates={"drift_limit": MDOF_DRIFT_LIMIT},
    )

    # modal analysis (for participation factor)
    copy_file(cfg["templates"]["run_modal"], folder / "run_modal.py")
    copy_analysis_config(
        cfg["templates"]["config_modal"],
        folder / "config_modal.py",
        results_folder_name="modal",
        update_config={"n_modes": get_n_primary_modes_from_design_file(design_dst)},
    )

    # # MSA files (roof-drift EDP for the MDOF)
    # add_msa_files(folder, site_idx, "ida_process_recorder_roof_drift")

    # cyclic pushover (FEMA 461)
    add_cyclic_pushover_files(folder, get_control_node_from_design_file(design_dst))
    return folder


mdof_folders = {ii: build_mdof_folder(ii) for ii in tqdm(
    [ii for ii in all_sites if designs[site_tag[ii]]["success"]],
    desc="Building MDOF folders")}
print(f"built {len(mdof_folders)} MDOF folders")

Building MDOF folders:   0%|          | 0/60 [00:00<?, ?it/s]

built 60 MDOF folders


## 4. Run modal analyses → `Gamma`, `m_star`

Each modal analysis runs in its own process (the per-folder `config_modal.py`
imports a folder-local `structural_model`, so they cannot share an interpreter).

In [8]:
run_modal_analyses_parallel(list(mdof_folders.values()), n_workers=N_WORKERS)

participation = {}
for ii, folder in tqdm(mdof_folders.items(), desc="Participation factors"):
    _, Gamma, m_star, gen_mass, eff_mass = convert_poc_to_eq_sdof_poc(
        folder / "modal", poc=None)
    participation[ii] = {"Gamma": float(Gamma), "m_star": float(m_star),
                         "generalised_mass": float(gen_mass),
                         "effective_mass": float(eff_mass)}

participation_path = Path(cfg["proc_data"]["site_specific_sdofs"]) / "participation_factors.json"
participation_path.parent.mkdir(parents=True, exist_ok=True)
with open(participation_path, "w") as f:
    json.dump(participation, f, indent=4)
print(f"saved participation factors for {len(participation)} sites -> {participation_path}")

Modal analyses:   0%|          | 0/60 [00:00<?, ?it/s]

Participation factors:   0%|          | 0/60 [00:00<?, ?it/s]

saved participation factors for 60 sites -> C:\Users\clemettn\Documents\phd\data_processed\10_site_specific_sdofs\participation_factors.json


### Load the participation factors

Loaded from the processed-data folder so the modal cell above can be **skipped** on re-runs (as long as `participation_factors.json` exists).

In [9]:
participation_path = Path(cfg["proc_data"]["site_specific_sdofs"]) / "participation_factors.json"
with open(participation_path) as f:
    participation = {int(k): v for k, v in json.load(f).items()}

pd.DataFrame(participation).T

,Gamma,m_star,generalised_mass,effective_mass
0,1.278449,216.846255,169.616626,277.226942
1,1.280521,214.073385,167.176721,274.125570
2,1.280521,214.073385,167.176721,274.125570
3,1.269607,220.508537,173.682546,279.959131
4,1.278449,216.846255,169.616626,277.226942
5,1.280521,214.073385,167.176721,274.125570
6,1.280521,214.073385,167.176721,274.125570
7,1.290718,207.146308,160.489184,267.367506
8,1.278449,216.846255,169.616626,277.226942
9,1.284847,210.762184,164.036775,270.797193


## 5. Design the equivalent SDOFs

`assemble_sdof_material_params(Vb_coeff, Wt, hi, aspect, mass=m_star, gamma=Gamma)`
turns each site's `Vb_coeff` into the OpenSees material parameters for the
two-spring SDOF, in SDOF coordinates.

In [10]:
sdof_out_dir = Path(cfg["proc_data"]["site_specific_sdofs"])
sdof_out_dir.mkdir(parents=True, exist_ok=True)

sdof_params = {}
for ii in tqdm(mdof_folders, desc="Designing SDOFs"):
    tag = site_tag[ii]
    d = designs[tag]
    p = participation[ii]
    raw = assemble_sdof_material_params(
        Vb_coeff=d["Vb_coeff"], Wt=d["Wt"], hi=STOREY_HEIGHT, aspect=ASPECT,
        mass=p["m_star"], gamma=p["Gamma"])
    params = _to_plain(raw)
    sdof_params[ii] = params

    record = {
        "tag": tag, "site_idx": ii, "S_alpha_RP": d["S_alpha_RP"],
        "Vb_coeff": d["Vb_coeff"], "Wt": d["Wt"], "T_mdof": d["T"],
        "Gamma": p["Gamma"], "m_star": p["m_star"], **params,
    }
    with open(sdof_out_dir / f"{tag}_sdof_props_from_parameterisation.json", "w") as f:
        json.dump(record, f, indent=4)

print(f"saved {len(sdof_params)} SDOF parameter files to {sdof_out_dir}")

Designing SDOFs:   0%|          | 0/60 [00:00<?, ?it/s]

saved 60 SDOF parameter files to C:\Users\clemettn\Documents\phd\data_processed\10_site_specific_sdofs


## 6. Build the SDOF analysis folders + stripe pickles

One SDOF folder per site at `DEST_ROOT/site_{ii}/sdof/`, sharing the same MSA
files (x-displacement EDP) but with the site's parameterised SDOF model.

In [ ]:
def build_sdof_folder(site_idx: int) -> Path:
    folder = DEST_ROOT / f"site_{site_idx}" / "sdof_param"
    folder.mkdir(parents=True, exist_ok=True)

    copy_structural_model(
        cfg["templates"]["model_cbf_sdof"],
        folder / "structural_model.py",
        ops_updates=sdof_params[site_idx],
        recorder_updates={"disp_limit": DISP_LIMIT},
        damping_updates={"damping_ratio": DAMPING_RATIO},
    )
    # add_msa_files(folder, site_idx, "ida_process_recorder_x_displacement")

    # modal analysis (n_modes=1, dense solver for the 2-node SDOF); for completeness
    copy_file(cfg["templates"]["run_modal"], folder / "run_modal.py")
    copy_analysis_config(
        cfg["templates"]["config_modal"],
        folder / "config_modal.py",
        results_folder_name="modal",
        update_config={"n_modes": 1, "solver": "-fullGenLapack"},
    )

    # cyclic pushover (FEMA 461); SDOF free node = ctrl node 2
    add_cyclic_pushover_files(folder, ctrl_node=2)
    return folder


sdof_folders = {ii: build_sdof_folder(ii) for ii in tqdm(
    list(sdof_params), desc="Building SDOF folders")}
print(f"built {len(sdof_folders)} SDOF folders")

Building SDOF folders:   0%|          | 0/60 [00:00<?, ?it/s]

built 60 SDOF folders


### Run the SDOF modal analyses

For completeness each `SDOF_param` folder also gets a modal analysis (`n_modes=1`, `-fullGenLapack`). Run them in isolated processes like the MDOFs.

In [13]:
result = run_modal_analyses_parallel(list(sdof_folders.values()), n_workers=N_WORKERS)

Modal analyses:   0%|          | 0/60 [00:00<?, ?it/s]

## 7. SDOF batch launcher (SDOFs only)

One generic batch file listing every `(site_{ii}/sdof/run_msa_site.py,
config_msa_{gmset}.py)` job. The MDOF MSA is intentionally **not** batched.

In [ ]:
batch_jobs = [
    {
        "script": folder / "run_msa_site.py",
        "config": [folder / f"config_msa_{g}.py" for g in GM_SETS],
    }
    for folder in sdof_folders.values()
]

batch_dst = Path(cfg["scripts"]["wp1pt4pt1_batch_run"]) / "sdof_param_msa.py"
batch_dst.parent.mkdir(parents=True, exist_ok=True)
configure_batch_run_file(cfg["templates"]["batch_run"], batch_dst, batch_jobs)
print(f"wrote SDOF MSA batch ({len(batch_jobs)} sites x {len(GM_SETS)} GM sets) -> {batch_dst}")

wrote SDOF MSA batch (60 sites x 2 GM sets) -> C:\Users\clemettn\Documents\phd\phd_project\scripts\WP1_ground_motion_set\batch_run_analysis_scripts\wp1pt4pt1\sdof_msa.py


## 8. Cyclic pushover batch launchers

One batch file for the MDOF cyclic pushovers and one for the SDOF (`SDOF_param`) cyclic pushovers. Distinct names so they don't clobber `wp1pt4pt1a`'s `mdof_cpo.py`.

In [ ]:
mdof_cpo_jobs = [{"script": f / "run_cyclic_pushover.py",
                  "config": [f / "config_cyclic_pushover.py"]} for f in mdof_folders.values()]
sdof_cpo_jobs = [{"script": f / "run_cyclic_pushover.py",
                  "config": [f / "config_cyclic_pushover.py"]} for f in sdof_folders.values()]

batch_root = Path(cfg["scripts"]["wp1pt4pt1_batch_run"])
batch_root.mkdir(parents=True, exist_ok=True)
configure_batch_run_file(cfg["templates"]["batch_run"], batch_root / "site_mdof_cpo.py", mdof_cpo_jobs)
configure_batch_run_file(cfg["templates"]["batch_run"], batch_root / "site_sdof_param_cpo.py", sdof_cpo_jobs)
print(f"wrote site_mdof_cpo.py ({len(mdof_cpo_jobs)} jobs) and "
      f"site_sdof_param_cpo.py ({len(sdof_cpo_jobs)} jobs) -> {batch_root}")